# LeetCode 1031 — Maximum Sum of Two Non-Overlapping Subarrays bằng Python

Link đề: https://leetcode.com/problems/maximum-sum-of-two-non-overlapping-subarrays/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def maxSumTwoNoOverlap(self, nums: List[int], firstLen: int, secondLen: int) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất**. Hai đoạn con (liên tiếp) có độ dài `firstLen` và `secondLen` **không giao nhau**; đoạn dài `firstLen` có thể nằm **trước hoặc sau** đoạn dài `secondLen`. Điểm cốt lõi: dùng **tổng tiền tố** để tính tổng cửa sổ trong `O(1)`; duyệt vị trí kết thúc của đoạn đứng **sau** và giữ **tổng cửa sổ lớn nhất của đoạn đứng trước** đã hoàn tất. Oracle độc lập: thử **mọi cặp vị trí bắt đầu**, tính tổng bằng `sum(nums[i:i+L])`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn hai cửa sổ liên tiếp, độ dài `firstLen` và `secondLen`, rời nhau (không chung phần tử), tối đa hoá tổng.
2. Nắm công cụ: tổng tiền tố, tổng cửa sổ trượt, cực đại tiền tố/hậu tố, bảng thưa (sparse table) cho truy vấn cực đại đoạn.
3. Đi từ **thử mọi cặp vị trí** → **một lượt với cực đại cửa sổ đứng trước (hai thứ tự)** → **mảng cực đại tiền tố/hậu tố** → **sparse table RMQ** → **DP theo trạng thái**.
4. Hiểu mấu chốt: với đoạn thứ hai kết thúc tại `j`, đoạn thứ nhất phải **kết thúc trước khi đoạn thứ hai bắt đầu**: chỉ cần cực đại tổng cửa sổ độ dài `L₁` trong `nums[0 .. j − L₂]`. Xét **cả hai thứ tự** (`first` trước `second` và ngược lại).
5. Tránh lỗi kinh điển: chỉ xét một thứ tự, cho phép giao nhau, tham lam "chọn cửa sổ tốt nhất rồi cửa sổ kia" (sai), chừa khe hở thừa (`+1`), lệch chỉ số tổng tiền tố, nhầm hai độ dài khi đổi thứ tự.

## 2. Đề bài & ví dụ

Cho mảng `nums` và hai số `firstLen`, `secondLen`. Trả về **tổng lớn nhất** của hai đoạn con **không giao nhau** có độ dài `firstLen` và `secondLen`.

Đoạn độ dài `firstLen` có thể ở trước hoặc sau đoạn độ dài `secondLen`, miễn là không giao nhau. Đoạn con là phần **liên tiếp** của mảng.

**Ví dụ 1:** `nums = [0,6,5,2,2,5,1,9,4]`, `firstLen = 1`, `secondLen = 2` → `20` (`[9]` và `[6,5]`).

**Ví dụ 2:** `nums = [3,8,1,3,2,1,8,9,0]`, `firstLen = 3`, `secondLen = 2` → `29` (`[3,8,1]` và `[8,9]`).

**Ví dụ 3:** `nums = [2,1,5,6,0,9,5,0,3,8]`, `firstLen = 4`, `secondLen = 3` → `31` (`[5,6,0,9]` và `[0,3,8]`).

**Ràng buộc:** `1 <= firstLen, secondLen <= 1000`; `2 <= firstLen + secondLen <= 1000`; `firstLen + secondLen <= nums.length <= 1000`; `0 <= nums[i] <= 1000`.

In [1]:
import itertools
import random
import time
from typing import List

examples = [
    (([0, 6, 5, 2, 2, 5, 1, 9, 4], 1, 2), 20), (([3, 8, 1, 3, 2, 1, 8, 9, 0], 3, 2), 29), (([2, 1, 5, 6, 0, 9, 5, 0, 3, 8], 4, 3), 31),
    (([1, 1], 1, 1), 2), (([5, 0, 0, 5], 2, 2), 10),
]
for (a, l1, l2), want in examples:
    print(f"nums = {a}, firstLen = {l1}, secondLen = {l2} -> {want}")

nums = [0, 6, 5, 2, 2, 5, 1, 9, 4], firstLen = 1, secondLen = 2 -> 20
nums = [3, 8, 1, 3, 2, 1, 8, 9, 0], firstLen = 3, secondLen = 2 -> 29
nums = [2, 1, 5, 6, 0, 9, 5, 0, 3, 8], firstLen = 4, secondLen = 3 -> 31
nums = [1, 1], firstLen = 1, secondLen = 1 -> 2
nums = [5, 0, 0, 5], firstLen = 2, secondLen = 2 -> 10


## 3. Kiến thức nền

### 3.1 Tổng tiền tố và tổng cửa sổ

`P[0] = 0`, `P[i] = nums[0] + … + nums[i−1]`. Tổng cửa sổ độ dài `L` bắt đầu tại `s` là `P[s + L] − P[s]` (`O(1)`).

In [2]:
nums = [0, 6, 5, 2, 2, 5, 1, 9, 4]
P = [0]
for x in nums:
    P.append(P[-1] + x)
print("P =", P)
print("cửa sổ độ dài 2 bắt đầu tại 1:", P[3] - P[1], "= nums[1] + nums[2] =", nums[1] + nums[2])

P = [0, 0, 6, 11, 13, 15, 20, 21, 30, 34]
cửa sổ độ dài 2 bắt đầu tại 1: 11 = nums[1] + nums[2] = 11


### 3.2 Hai thứ tự

Hai đoạn không giao nhau thì **một** đoạn nằm hoàn toàn trước đoạn kia. Có hai khả năng: đoạn `firstLen` trước đoạn `secondLen`, hoặc ngược lại. Ta tính tốt nhất cho **từng thứ tự** rồi lấy max.

### 3.3 Điều kiện không giao nhau

Cửa sổ `A` độ dài `a` bắt đầu tại `i` (chiếm `[i, i + a − 1]`), cửa sổ `B` độ dài `b` bắt đầu tại `j`. `A` trước `B` không giao khi `i + a ≤ j`, tức `A` kết thúc (chỉ số cuối `i + a − 1`) **trước** vị trí bắt đầu của `B`.

In [3]:
def window_sum(P, s, L):
    return P[s + L] - P[s]


print(window_sum(P, 1, 2), window_sum(P, 7, 1), "| A=[7..7] (a=1) trước B=[1..2]? không; B=[1..2] trước A=[7]: 1+2 <= 7:", 1 + 2 <= 7)

11 9 | A=[7..7] (a=1) trước B=[1..2]? không; B=[1..2] trước A=[7]: 1+2 <= 7: True


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `nums = [0,6,5,2,2,5,1,9,4]`, `firstLen = 1`, `secondLen = 2`. Cửa sổ độ dài 2 tốt nhất: `[6,5] = 11` (vị trí `1`); cửa sổ độ dài 1 tốt nhất không giao nó: `[9] = 9` ⇒ `20`. Nhưng ví dụ cho thấy ta **không** thể chọn tham lam (cửa sổ tốt nhất mỗi loại) vì chúng có thể giao nhau; cần xét đồng thời.

Cách chuẩn: duyệt vị trí kết thúc của đoạn đứng sau; giữ cực đại tổng của đoạn đứng trước trong phần mảng đã "đóng".

## 5. Brute force — thử mọi cặp vị trí bắt đầu

**Ý tưởng:** duyệt mọi `i` (đầu đoạn `firstLen`) và `j` (đầu đoạn `secondLen`); nếu hai đoạn không giao (`i + L₁ ≤ j` hoặc `j + L₂ ≤ i`) thì tính tổng (bằng `sum(nums[i:i+L₁]) + sum(nums[j:j+L₂])` — đúng định nghĩa) và lấy max. `O(n² · (L₁ + L₂))`; với tổng tiền tố `O(n²)`. Dùng làm **oracle**.

In [4]:
def two_brute(nums, l1, l2):
    n = len(nums)
    best = -1
    for i in range(n - l1 + 1):
        for j in range(n - l2 + 1):
            if i + l1 <= j or j + l2 <= i:                      # không giao nhau
                best = max(best, sum(nums[i:i + l1]) + sum(nums[j:j + l2]))
    return best


for (a, l1, l2), want in examples:
    assert two_brute(a, l1, l2) == want
print([two_brute(a, l1, l2) for (a, l1, l2), _ in examples])

[20, 29, 31, 2, 10]


In [5]:
# Code nộp LeetCode cho Brute force (O(n^2) với tổng tiền tố)
class SolutionBrute:
    def maxSumTwoNoOverlap(self, nums: List[int], firstLen: int, secondLen: int) -> int:
        n = len(nums)
        P = [0]
        for x in nums:
            P.append(P[-1] + x)
        best = 0
        for i in range(n - firstLen + 1):
            for j in range(n - secondLen + 1):
                if i + firstLen <= j or j + secondLen <= i:
                    best = max(best, P[i + firstLen] - P[i] + P[j + secondLen] - P[j])
        return best

**Độ phức tạp:** `O(n²)` thời gian (`≤ 10^6` với `n = 1000`), `O(n)` bộ nhớ.

**Nút thắt:** ta so mọi cặp đầu đoạn trong khi với mỗi đoạn đứng sau chỉ cần cực đại của đoạn đứng trước — một giá trị duy trì được.

## 6. Bước đột phá

Với thứ tự "`A` (độ dài `a`) trước `B` (độ dài `b`)": duyệt vị trí kết thúc `j` của `B` (từ `a + b`). Đoạn `A` phải nằm trong `nums[0 .. j − b]`; giữ `best_a` = cực đại tổng cửa sổ độ dài `a` kết thúc tại hoặc trước `j − b`. Đáp án thứ tự này `= max_j (best_a + tổng B kết thúc tại j)`. Làm tương tự với thứ tự ngược lại; lấy max.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** một lượt với `best_a` (đáp án chính), **Cách 3** mảng cực đại tiền tố/hậu tố, **Cách 4** sparse table (RMQ), **Cách 5** DP theo trạng thái.

### Cách 2 — Một lượt với cực đại cửa sổ đứng trước

**Giải thích từng đoạn:** hàm `solve(a, b)` tính tốt nhất khi đoạn độ dài `a` đứng trước đoạn độ dài `b`; đáp án là `max(solve(L₁, L₂), solve(L₂, L₁))`.

1. `P` là tổng tiền tố.
2. `best_a = 0`, `ans = 0`.
3. Với `j` từ `a + b` tới `n` (`j` là vị trí **sau** cuối đoạn `B`, tức `B = nums[j−b .. j−1]`): đoạn `A` kết thúc tại `j − b` (cửa sổ `nums[j−b−a .. j−b−1]`) vừa trở nên hợp lệ ⇒ `best_a = max(best_a, P[j−b] − P[j−b−a])`; rồi `ans = max(ans, best_a + P[j] − P[j−b])`.

In [6]:
def two_one_pass(nums, l1, l2):
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    n = len(nums)

    def solve(a, b):                                       # đoạn độ dài a đứng TRƯỚC đoạn độ dài b
        best_a, ans = 0, 0
        for j in range(a + b, n + 1):                      # B = nums[j-b : j]
            best_a = max(best_a, P[j - b] - P[j - b - a])  # A kết thúc tại j - b (không giao B)
            ans = max(ans, best_a + P[j] - P[j - b])
        return ans

    return max(solve(l1, l2), solve(l2, l1))


for (a, l1, l2), want in examples:
    assert two_one_pass(a, l1, l2) == want
print([two_one_pass(a, l1, l2) for (a, l1, l2), _ in examples])

[20, 29, 31, 2, 10]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionOnePass:
    def maxSumTwoNoOverlap(self, nums: List[int], firstLen: int, secondLen: int) -> int:
        n = len(nums)
        P = [0]
        for x in nums:
            P.append(P[-1] + x)

        def solve(a, b):
            best_a = ans = 0
            for j in range(a + b, n + 1):
                best_a = max(best_a, P[j - b] - P[j - b - a])
                ans = max(ans, best_a + P[j] - P[j - b])
            return ans

        return max(solve(firstLen, secondLen), solve(secondLen, firstLen))

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (tổng tiền tố).

### Cách 3 — Mảng cực đại tiền tố / hậu tố của tổng cửa sổ

**Ý tưởng:** tính `W_a[s]` (tổng cửa sổ độ dài `a` bắt đầu tại `s`). `prefix_a[k]` = cực đại `W_a[s]` với cửa sổ kết thúc trong `[0, k)`; `suffix_b[k]` = cực đại `W_b[s]` với cửa sổ bắt đầu từ `≥ k`. Với mỗi ranh giới `k`: `prefix_a[k] + suffix_b[k]`; xét cả hai thứ tự.

In [8]:
def two_prefix_suffix(nums, l1, l2):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)

    def order(a, b):                                        # a trước b: tách tại ranh giới k (A trong [0,k), B trong [k,n))
        NEG = float("-inf")
        best_before = [NEG] * (n + 1)                       # best_before[k]: max tổng cửa sổ độ dài a nằm hẳn trong [0, k)
        for k in range(a, n + 1):
            best_before[k] = max(best_before[k - 1], P[k] - P[k - a])
        best_after = [NEG] * (n + 2)                        # best_after[k]: max tổng cửa sổ độ dài b nằm hẳn trong [k, n)
        for k in range(n - b, -1, -1):
            best_after[k] = max(best_after[k + 1], P[k + b] - P[k])
        return max(best_before[k] + best_after[k] for k in range(a, n - b + 1))

    return max(order(l1, l2), order(l2, l1))


for (a, l1, l2), want in examples:
    assert two_prefix_suffix(a, l1, l2) == want
print([two_prefix_suffix(a, l1, l2) for (a, l1, l2), _ in examples])

[20, 29, 31, 2, 10]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionPrefixSuffix:
    def maxSumTwoNoOverlap(self, nums: List[int], firstLen: int, secondLen: int) -> int:
        n = len(nums)
        P = [0]
        for x in nums:
            P.append(P[-1] + x)
        NEG = float("-inf")

        def order(a, b):
            before = [NEG] * (n + 1)
            for k in range(a, n + 1):
                before[k] = max(before[k - 1], P[k] - P[k - a])
            after = [NEG] * (n + 2)
            for k in range(n - b, -1, -1):
                after[k] = max(after[k + 1], P[k + b] - P[k])
            return max(before[k] + after[k] for k in range(a, n - b + 1))

        return max(order(firstLen, secondLen), order(secondLen, firstLen))

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### Cách 4 — Sparse table (truy vấn cực đại đoạn)

**Ý tưởng:** dựng `W_a`, `W_b`; với mỗi vị trí bắt đầu `i` của đoạn `A` (độ dài `a`), cửa sổ `B` hợp lệ có đầu `j ∈ [i + a, n − b]` (nếu `A` trước) — lấy `max W_b[j]` trên khoảng đó bằng **bảng thưa** (`O(1)` mỗi truy vấn sau tiền xử lý `O(n log n)`). Xét cả hai thứ tự.

In [10]:
def build_sparse(arr):
    table = [arr[:]]
    k = 1
    while (1 << k) <= len(arr):
        prev = table[-1]
        half = 1 << (k - 1)
        table.append([max(prev[i], prev[i + half]) for i in range(len(arr) - (1 << k) + 1)])
        k += 1
    return table


def range_max(table, lo, hi):                            # max arr[lo..hi] (đóng)
    k = (hi - lo + 1).bit_length() - 1
    return max(table[k][lo], table[k][hi - (1 << k) + 1])


def two_sparse(nums, l1, l2):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    W = lambda L: [P[s + L] - P[s] for s in range(n - L + 1)]

    def order(a, b):                                      # a trước b
        Wa, Wb = W(a), W(b)
        table = build_sparse(Wb)
        best = -1
        for i in range(len(Wa)):
            lo, hi = i + a, n - b                         # đầu của B nằm trong [i + a, n - b]
            if lo <= hi:
                best = max(best, Wa[i] + range_max(table, lo, hi))
        return best

    return max(order(l1, l2), order(l2, l1))


for (a, l1, l2), want in examples:
    assert two_sparse(a, l1, l2) == want
print([two_sparse(a, l1, l2) for (a, l1, l2), _ in examples])

[20, 29, 31, 2, 10]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionSparse:
    def maxSumTwoNoOverlap(self, nums: List[int], firstLen: int, secondLen: int) -> int:
        n = len(nums)
        P = [0]
        for x in nums:
            P.append(P[-1] + x)

        def build(arr):
            table = [arr[:]]
            k = 1
            while (1 << k) <= len(arr):
                prev, half = table[-1], 1 << (k - 1)
                table.append([max(prev[i], prev[i + half]) for i in range(len(arr) - (1 << k) + 1)])
                k += 1
            return table

        def qmax(table, lo, hi):
            k = (hi - lo + 1).bit_length() - 1
            return max(table[k][lo], table[k][hi - (1 << k) + 1])

        def order(a, b):
            Wa = [P[s + a] - P[s] for s in range(n - a + 1)]
            Wb = [P[s + b] - P[s] for s in range(n - b + 1)]
            table = build(Wb)
            best = 0
            for i in range(len(Wa)):
                lo, hi = i + a, n - b
                if lo <= hi:
                    best = max(best, Wa[i] + qmax(table, lo, hi))
            return best

        return max(order(firstLen, secondLen), order(secondLen, firstLen))

**Độ phức tạp:** `O(n log n)` thời gian (dựng bảng thưa), `O(n log n)` bộ nhớ.

### Cách 5 — DP theo trạng thái trên vị trí kết thúc

**Ý tưởng:** duyệt `i` từ `1` tới `n` (độ dài tiền tố đã xét). Giữ các đại lượng: `A[i]` = cực đại tổng cửa sổ `L₁` kết thúc `≤ i`; `B[i]` tương tự với `L₂`; `AB[i]` = cực đại (`A` rồi `B`) với `B` kết thúc `≤ i`: `AB[i] = max(AB[i−1], A[i − L₂] + sum(B kết thúc tại i))`; `BA[i]` đối xứng. Đáp án `max(AB[n], BA[n])`.

In [12]:
def two_dp(nums, l1, l2):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    NEG = float("-inf")
    A = [NEG] * (n + 1)          # A[i]: max tổng cửa sổ độ dài l1 kết thúc <= i
    B = [NEG] * (n + 1)
    AB = [NEG] * (n + 1)         # đoạn l1 rồi đoạn l2, đoạn l2 kết thúc <= i
    BA = [NEG] * (n + 1)
    for i in range(1, n + 1):
        A[i] = A[i - 1]
        B[i] = B[i - 1]
        AB[i] = AB[i - 1]
        BA[i] = BA[i - 1]
        if i >= l1:
            s1 = P[i] - P[i - l1]
            A[i] = max(A[i], s1)
            if i - l1 >= l2 and B[i - l1] > NEG:
                BA[i] = max(BA[i], B[i - l1] + s1)          # đoạn l1 kết thúc tại i, đoạn l2 nằm trước
        if i >= l2:
            s2 = P[i] - P[i - l2]
            B[i] = max(B[i], s2)
            if i - l2 >= l1 and A[i - l2] > NEG:
                AB[i] = max(AB[i], A[i - l2] + s2)          # đoạn l2 kết thúc tại i, đoạn l1 nằm trước
    return max(AB[n], BA[n])


for (a, l1, l2), want in examples:
    assert two_dp(a, l1, l2) == want
print([two_dp(a, l1, l2) for (a, l1, l2), _ in examples])

[20, 29, 31, 2, 10]


In [13]:
# Code nộp LeetCode cho Cách 5
class SolutionDP:
    def maxSumTwoNoOverlap(self, nums: List[int], firstLen: int, secondLen: int) -> int:
        n = len(nums)
        P = [0]
        for x in nums:
            P.append(P[-1] + x)
        NEG = float("-inf")
        A, B, AB, BA = [NEG] * (n + 1), [NEG] * (n + 1), [NEG] * (n + 1), [NEG] * (n + 1)
        for i in range(1, n + 1):
            A[i], B[i], AB[i], BA[i] = A[i - 1], B[i - 1], AB[i - 1], BA[i - 1]
            if i >= firstLen:
                s1 = P[i] - P[i - firstLen]
                A[i] = max(A[i], s1)
                if i - firstLen >= secondLen:
                    BA[i] = max(BA[i], B[i - firstLen] + s1)
            if i >= secondLen:
                s2 = P[i] - P[i - secondLen]
                B[i] = max(B[i], s2)
                if i - secondLen >= firstLen:
                    AB[i] = max(AB[i], A[i - secondLen] + s2)
        return max(AB[n], BA[n])

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với `nums = [0,6,5,2,2,5,1,9,4]`, `firstLen = 1`, `secondLen = 2` (`P = [0,0,6,11,13,15,20,21,30,34]`). Thứ tự "`a = 1` trước `b = 2`" (đoạn dài 1 trước đoạn dài 2):

| `j` | `B = nums[j−2..j−1]` | `A` mới hợp lệ (kết thúc tại `j−2`) | `best_a` | tổng `best_a + B` |
|---|---|---|---|---|
| 3 | `nums[1..2] = [6,5]` = 11 | `nums[0] = 0` | 0 | 11 |
| 4 | `[5,2]` = 7 | `nums[1] = 6` | 6 | 13 |
| 5 | `[2,2]` = 4 | `nums[2] = 5` | 6 | 10 |
| 6 | `[2,5]` = 7 | `nums[3] = 2` | 6 | 13 |
| 7 | `[5,1]` = 6 | `nums[4] = 2` | 6 | 12 |
| 8 | `[1,9]` = 10 | `nums[5] = 5` | 6 | 16 |
| 9 | `[9,4]` = 13 | `nums[6] = 1` | 6 | 19 |

Thứ tự "`a = 2` trước `b = 1`" cho `[6,5] = 11` rồi `[9] = 9` ⇒ `20` ✓. Hàm trace in từng bước cho **hai thứ tự**:

In [14]:
def trace_one_pass(nums, l1, l2):
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    n = len(nums)
    results = []
    for a, b, label in ((l1, l2, "first trước second"), (l2, l1, "second trước first")):
        print(f"thứ tự {label}: A dài {a}, B dài {b}")
        best_a, ans = 0, 0
        for j in range(a + b, n + 1):
            sa = P[j - b] - P[j - b - a]
            best_a = max(best_a, sa)
            sb = P[j] - P[j - b]
            ans = max(ans, best_a + sb)
            print(f"  j={j}: A kết thúc tại {j - b} (tổng {sa}), best_a={best_a}, B=nums[{j - b}:{j}] tổng {sb} -> {best_a + sb}")
        results.append(ans)
    print("=>", max(results), "(hai thứ tự:", results, ")")
    return max(results)


assert trace_one_pass([0, 6, 5, 2, 2, 5, 1, 9, 4], 1, 2) == 20

thứ tự first trước second: A dài 1, B dài 2
  j=3: A kết thúc tại 1 (tổng 0), best_a=0, B=nums[1:3] tổng 11 -> 11
  j=4: A kết thúc tại 2 (tổng 6), best_a=6, B=nums[2:4] tổng 7 -> 13
  j=5: A kết thúc tại 3 (tổng 5), best_a=6, B=nums[3:5] tổng 4 -> 10
  j=6: A kết thúc tại 4 (tổng 2), best_a=6, B=nums[4:6] tổng 7 -> 13
  j=7: A kết thúc tại 5 (tổng 2), best_a=6, B=nums[5:7] tổng 6 -> 12
  j=8: A kết thúc tại 6 (tổng 5), best_a=6, B=nums[6:8] tổng 10 -> 16
  j=9: A kết thúc tại 7 (tổng 1), best_a=6, B=nums[7:9] tổng 13 -> 19
thứ tự second trước first: A dài 2, B dài 1
  j=3: A kết thúc tại 2 (tổng 6), best_a=6, B=nums[2:3] tổng 5 -> 11
  j=4: A kết thúc tại 3 (tổng 11), best_a=11, B=nums[3:4] tổng 2 -> 13
  j=5: A kết thúc tại 4 (tổng 7), best_a=11, B=nums[4:5] tổng 2 -> 13
  j=6: A kết thúc tại 5 (tổng 4), best_a=11, B=nums[5:6] tổng 5 -> 16
  j=7: A kết thúc tại 6 (tổng 7), best_a=11, B=nums[6:7] tổng 1 -> 12
  j=8: A kết thúc tại 7 (tổng 6), best_a=11, B=nums[7:8] tổng 9 -> 20
  j=9:

**Trace vì sao cần cả hai thứ tự:** `nums = [5, 5, 5, 0, 0, 1]`, `firstLen = 1`, `secondLen = 3`: tốt nhất là đoạn dài 3 `[5,5,5]` đứng **trước** đoạn dài 1 `[1]` (`16`); thứ tự "`firstLen` trước" (đoạn dài 1 rồi đoạn dài 3) chỉ được `[5]` + `[5,5,0]` = `15` (xem Lỗi 1 mục 13).

## 9. Code nộp LeetCode

Chọn **Cách 2** (một lượt). Giữ nguyên `class Solution`, `maxSumTwoNoOverlap(self, nums, firstLen, secondLen)`:

In [15]:
class Solution:
    def maxSumTwoNoOverlap(self, nums: List[int], firstLen: int, secondLen: int) -> int:
        n = len(nums)
        P = [0]
        for x in nums:
            P.append(P[-1] + x)                           # tổng tiền tố

        def solve(a, b):                                  # đoạn độ dài a đứng TRƯỚC đoạn độ dài b
            best_a = ans = 0
            for j in range(a + b, n + 1):                 # B = nums[j - b : j]
                best_a = max(best_a, P[j - b] - P[j - b - a])   # A kết thúc tại j - b: không giao B
                ans = max(ans, best_a + P[j] - P[j - b])
            return ans

        return max(solve(firstLen, secondLen), solve(secondLen, firstLen))   # cả hai thứ tự


print([Solution().maxSumTwoNoOverlap(list(a), l1, l2) for (a, l1, l2), _ in examples])
assert all(Solution().maxSumTwoNoOverlap(list(a), l1, l2) == w for (a, l1, l2), w in examples)

[20, 29, 31, 2, 10]


## 10. Vì sao đúng

**(a) Hai đoạn rời nhau ⇒ một đoạn nằm hẳn trước.** Hai cửa sổ `[i, i + L₁ − 1]` và `[j, j + L₂ − 1]` không giao ⇔ `i + L₁ ≤ j` hoặc `j + L₂ ≤ i`. Vậy đáp án `= max(F, S)` với `F` = tốt nhất khi đoạn `firstLen` đứng trước, `S` = khi đoạn `secondLen` đứng trước. Hai trường hợp bao phủ mọi cặp hợp lệ.

**(b) Một lượt cho thứ tự "`a` trước `b`".** Với đoạn `B` kết thúc tại `j` (chiếm `[j − b, j − 1]`), đoạn `A` hợp lệ phải kết thúc không quá `j − b` (độ dài tiền tố `≤ j − b`), tức mọi cửa sổ `A` có `e = s + a ≤ j − b`. Gọi `best_a(j) = max_{e ≤ j − b} (P[e] − P[e − a])`. Khi `j` tăng `1`, `j − b` tăng `1`: chỉ thêm cửa sổ `A` kết thúc tại `j − b` — đúng bước cập nhật. Tốt nhất với `B` kết thúc tại `j` là `best_a(j) + (P[j] − P[j − b])`; lấy max theo `j`. Cửa sổ `A` đầu tiên khả dụng khi `j − b ≥ a`, tức `j ≥ a + b` (vòng lặp bắt đầu từ `a + b`).

**(c) Cách 3.** `before[k] = best_a(k)`, `after[k] = max_{s ≥ k} W_b[s]`: với ranh giới `k` (đoạn `A` hoàn toàn trong `[0, k)`, đoạn `B` hoàn toàn trong `[k, n)`), tốt nhất là `before[k] + after[k]`; mọi cặp `A` trước `B` không giao ứng với ít nhất một ranh giới (chọn `k` = vị trí kết thúc của `A`), nên max trên `k` bao phủ tất cả và không vượt quá tối ưu.

**(d) Cách 4.** Với `A` bắt đầu tại `i`, các `B` hợp lệ có đầu `j ≥ i + a` và `j ≤ n − b`; `range_max` cho `max W_b[j]` trong khoảng đó.

**(e) Cách 5.** `AB[i]` là max trên các cặp (`A` trước `B`) với `B` kết thúc `≤ i`; nhận `A[i − L₂]` (cực đại cửa sổ `A` kết thúc `≤ i − L₂`) cộng tổng `B` kết thúc tại `i`.

**(f) Tại sao tham lam "mỗi loại chọn tốt nhất" sai.** Hai cửa sổ tốt nhất của mỗi loại có thể **giao nhau**, khi đó phải bỏ bớt ở một bên; phải xét cặp đồng thời.

**(g) Kiểm chứng bằng máy.** Mọi cách khớp oracle `two_brute` (định nghĩa, `sum` trên lát cắt) trên **mọi** mảng `n ≤ 7` giá trị `{0,1,2}` và `n ≤ 6` giá trị `{0,1,2,3}` với mọi `(L₁, L₂)` hợp lệ; khớp brute `O(n²)` (tổng tiền tố) trên mảng ngẫu nhiên tới `n = 300`; các cách `O(n)` khớp nhau tới `n = 1000` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi cặp vị trí | `O(n²)` | `O(n)` | oracle |
| 2. Một lượt `best_a` | `O(n)` | `O(n)` | **đáp án chính** |
| 3. Cực đại tiền tố/hậu tố | `O(n)` | `O(n)` | dạng khai báo |
| 4. Sparse table | `O(n log n)` | `O(n log n)` | mở rộng cho truy vấn khác |
| 5. DP trạng thái | `O(n)` | `O(n)` | khung cho nhiều đoạn hơn |

Đo thực tế (`n = 1000`):

In [16]:
def bench(fn, args, repeat=20):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(*args)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(1031)
big = [rng.randint(0, 1000) for _ in range(1000)]
for name, fn in [("one_pass", two_one_pass), ("prefix_suffix", two_prefix_suffix), ("sparse", two_sparse), ("dp", two_dp)]:
    print(f"{name:14s}: {bench(fn, (big, 300, 400)) * 1e3:7.2f} ms -> {fn(big, 300, 400)}")
print(f"{'brute (n=200)':14s}: {bench(two_brute, (big[:200], 30, 40), repeat=1) * 1e3:7.1f} ms")

one_pass      :    0.20 ms -> 354872
prefix_suffix :    0.46 ms -> 354872


sparse        :    0.93 ms -> 354872
dp            :    0.34 ms -> 354872


brute (n=200) :    16.3 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mảng nhỏ với oracle định nghĩa, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn `n = 1000`.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "one_pass": two_one_pass,
    "prefix_suffix": two_prefix_suffix,
    "sparse": two_sparse,
    "dp": two_dp,
    "brute": two_brute,
    "Solution": lambda a, x, y: Solution().maxSumTwoNoOverlap(list(a), x, y),
    "SolutionOnePass": lambda a, x, y: SolutionOnePass().maxSumTwoNoOverlap(list(a), x, y),
    "SolutionPrefixSuffix": lambda a, x, y: SolutionPrefixSuffix().maxSumTwoNoOverlap(list(a), x, y),
    "SolutionSparse": lambda a, x, y: SolutionSparse().maxSumTwoNoOverlap(list(a), x, y),
    "SolutionDP": lambda a, x, y: SolutionDP().maxSumTwoNoOverlap(list(a), x, y),
    "SolutionBrute": lambda a, x, y: SolutionBrute().maxSumTwoNoOverlap(list(a), x, y),
}
SLOW = {"brute", "SolutionBrute"}
fast = [n for n in IMPLS if n not in SLOW]


def check(a, l1, l2, want=None, brute_n=300):
    if want is None and len(a) <= brute_n:
        want = two_brute(a, l1, l2) if len(a) <= 40 else SolutionBrute().maxSumTwoNoOverlap(a, l1, l2)
    results = {}
    for name in IMPLS:
        if name in SLOW and len(a) > brute_n:
            continue
        results[name] = IMPLS[name](a, l1, l2)
    assert len(set(results.values())) == 1, (a[:8], l1, l2, results)
    got = next(iter(results.values()))
    if want is not None:
        assert got == want, (a[:8], l1, l2, got, want)
    return got


for (a, l1, l2), want in examples:
    check(a, l1, l2, want)
print("ví dụ đề: oracle định nghĩa và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle định nghĩa và 11 cài đặt đúng


### B. Edge case

- `n = L₁ + L₂` (chỉ có một cách chia): tổng toàn mảng.
- `L₁ = L₂ = 1`: tổng hai phần tử lớn nhất ở hai vị trí khác nhau.
- Toàn số `0`; toàn số bằng nhau.
- Đoạn dài đứng đầu / cuối (cần cả hai thứ tự): `[5, 5, 5, 0, 0, 1]`, `L₁ = 1`, `L₂ = 3`.
- Cửa sổ tốt nhất của hai độ dài giao nhau (phải bỏ một): `[0, 10, 10, 0, 0, 1]`.
- `L₁ ≠ L₂` và hoán đổi `(L₁, L₂)` cho cùng đáp án.
- Giá trị lớn `1000` toàn bộ.

In [18]:
edge = [
    ([1, 2], 1, 1), ([1, 2, 3], 1, 2), ([1, 2, 3], 2, 1), ([5, 5, 5, 0, 0, 1], 1, 3), ([5, 5, 5, 0, 0, 1], 3, 1), ([0, 10, 10, 0, 0, 1], 2, 1), ([0, 10, 10, 0, 0, 1], 1, 2),
    ([0] * 8, 3, 2), ([7] * 8, 3, 2), ([1000] * 10, 4, 6), ([9, 0, 0, 0, 9], 1, 1), ([9, 0, 0, 0, 9], 2, 2), ([1, 9, 1, 9, 1, 9], 2, 2), ([4, 8, 1, 7, 3, 9, 2], 3, 2),
    ([0, 6, 5, 2, 2, 5, 1, 9, 4], 2, 1), ([0, 6, 5, 2, 2, 5, 1, 9, 4], 1, 2),
]
for a, l1, l2 in edge:
    check(a, l1, l2)
    check(a, l2, l1)                                                         # hoán đổi (firstLen, secondLen) không đổi đáp án
n = 500
check([1000] * n, 100, 400)
assert Solution().maxSumTwoNoOverlap([1000] * n, 100, 400) == 500000
print(len(edge), "edge case (và hoán đổi độ dài) + n = 500 toàn 1000: đúng")

16 edge case (và hoán đổi độ dài) + n = 500 toàn 1000: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_one_order(nums, l1, l2):                        # chỉ xét firstLen đứng trước secondLen
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    n = len(nums)
    best_a = ans = 0
    for j in range(l1 + l2, n + 1):
        best_a = max(best_a, P[j - l2] - P[j - l2 - l1])
        ans = max(ans, best_a + P[j] - P[j - l2])
    return ans


def bug_overlap_allowed(nums, l1, l2):                  # hai cửa sổ tốt nhất độc lập (có thể giao nhau)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    n = len(nums)
    return max(P[s + l1] - P[s] for s in range(n - l1 + 1)) + max(P[s + l2] - P[s] for s in range(n - l2 + 1))


def bug_greedy_two_best(nums, l1, l2):                  # chọn cửa sổ l1 tốt nhất rồi cửa sổ l2 tốt nhất không giao (tham lam)
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    i = max(range(n - l1 + 1), key=lambda s: P[s + l1] - P[s])
    best2 = -1
    for j in range(n - l2 + 1):
        if i + l1 <= j or j + l2 <= i:
            best2 = max(best2, P[j + l2] - P[j])
    return P[i + l1] - P[i] + best2


def bug_gap(nums, l1, l2):                              # chừa khe hở thừa (+1)
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    best = 0
    for i in range(n - l1 + 1):
        for j in range(n - l2 + 1):
            if i + l1 < j or j + l2 < i:                # BUG: thiếu dấu bằng khi chạm (j == i + l1)
                best = max(best, P[i + l1] - P[i] + P[j + l2] - P[j])
    return best


def bug_prefix_off_by_one(nums, l1, l2):                # tổng cửa sổ lệch một chỉ số
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    n = len(nums)

    def solve(a, b):
        best_a = ans = 0
        for j in range(a + b, n + 1):
            best_a = max(best_a, P[j - b] - P[j - b - a + 1] if j - b - a + 1 >= 0 else 0)     # BUG: cửa sổ dài a - 1
            ans = max(ans, best_a + P[j] - P[j - b])
        return ans

    return max(solve(l1, l2), solve(l2, l1))


def bug_not_swapped_lengths(nums, l1, l2):              # thứ tự ngược nhưng quên đổi độ dài
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    n = len(nums)

    def solve(a, b):
        best_a = ans = 0
        for j in range(a + b, n + 1):
            best_a = max(best_a, P[j - b] - P[j - b - a])
            ans = max(ans, best_a + P[j] - P[j - b])
        return ans

    return max(solve(l1, l2), solve(l1, l2))


BUGS = [bug_one_order, bug_overlap_allowed, bug_greedy_two_best, bug_gap, bug_prefix_off_by_one, bug_not_swapped_lengths]
pool = [(list(a), l1, l2) for n in range(2, 7) for a in itertools.product(range(4), repeat=n) for l1 in range(1, n) for l2 in range(1, n) if l1 + l2 <= n]
for bug in BUGS:
    found = None
    for a, l1, l2 in pool:
        try:
            got = bug(a, l1, l2)
        except Exception as e:
            got = type(e).__name__
        if got != two_brute(a, l1, l2):
            found = (a, l1, l2, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:26s} sai trên nums={found[0]}, ({found[1]}, {found[2]}): ra {found[3]}, đúng {two_brute(*found[:3])}")

bug_one_order              sai trên nums=[1, 0, 1, 1], (2, 1): ra 2, đúng 3
bug_overlap_allowed        sai trên nums=[0, 1], (1, 1): ra 2, đúng 1
bug_greedy_two_best        sai trên nums=[0, 1, 0], (1, 2): ra 0, đúng 1
bug_gap                    sai trên nums=[0, 1], (1, 1): ra 0, đúng 1
bug_prefix_off_by_one      sai trên nums=[1, 0], (1, 1): ra 0, đúng 1
bug_not_swapped_lengths    sai trên nums=[1, 0, 1, 1], (2, 1): ra 2, đúng 3


### D. Vét cạn mảng nhỏ

Oracle `two_brute` (mọi cặp vị trí, tổng bằng `sum` trên lát cắt, định nghĩa). Mọi mảng độ dài `2..7` với giá trị `{0,1,2}` (`3^2 + … + 3^7 = 3 249` mảng) và độ dài `2..6` với giá trị `{0,1,2,3}` (`4^2 + … + 4^6 = 5 456` mảng), mỗi mảng với **mọi** `(L₁, L₂)`, `L₁, L₂ ≥ 1`, `L₁ + L₂ ≤ n`.

In [20]:
t0 = time.perf_counter()
cnt = 0
main_impls = ["one_pass", "prefix_suffix", "sparse", "dp", "Solution", "SolutionOnePass", "SolutionPrefixSuffix", "SolutionSparse", "SolutionDP", "SolutionBrute"]
for hi_val, max_n in ((3, 7), (4, 6)):
    for n in range(2, max_n + 1):
        for arr in itertools.product(range(hi_val), repeat=n):
            a = list(arr)
            for l1 in range(1, n):
                for l2 in range(1, n - l1 + 1):
                    want = two_brute(a, l1, l2)
                    for name in main_impls:
                        assert IMPLS[name](a, l1, l2) == want, (name, a, l1, l2, IMPLS[name](a, l1, l2), want)
                    cnt += 1
print(cnt, "ca (mảng ≤ 7 phần tử trong {0,1,2}; ≤ 6 trong {0..3}; mọi (L1, L2)):", len(main_impls), "cài đặt khớp oracle định nghĩa;", round(time.perf_counter() - t0, 1), "s")

133292 ca (mảng ≤ 7 phần tử trong {0,1,2}; ≤ 6 trong {0..3}; mọi (L1, L2)): 10 cài đặt khớp oracle định nghĩa; 7.0 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Các cách khớp nhau; với `n ≤ 300` khớp brute `O(n²)`; (2) hoán đổi `(firstLen, secondLen)` không đổi đáp án; (3) đảo mảng không đổi đáp án; (4) cộng hằng `c` vào mọi phần tử tăng đáp án `c·(L₁ + L₂)`; (5) nhân mọi phần tử với `k` nhân đáp án với `k`; (6) đáp án `≤` tổng hai cửa sổ tốt nhất độc lập (có thể giao nhau) và `≥` mỗi thứ tự cố định; (7) có cặp cửa sổ cụ thể đạt đáp án (dựng lại bằng tìm kiếm).

In [21]:
def best_positions(nums, l1, l2):
    """Tìm một cặp vị trí đầu (i, j) đạt đáp án (O(n²) trên tổng tiền tố)."""
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    best, pos = -1, None
    for i in range(n - l1 + 1):
        for j in range(n - l2 + 1):
            if i + l1 <= j or j + l2 <= i:
                s = P[i + l1] - P[i] + P[j + l2] - P[j]
                if s > best:
                    best, pos = s, (i, j)
    return best, pos


rng = random.Random(20240118)
t0 = time.perf_counter()
for it in range(600):
    n = rng.choice([2, 3, 5, 10, 30, 100, 300, 1000])
    hi = rng.choice([1, 5, 1000])
    a = [rng.randint(0, hi) for _ in range(n)]
    l1 = rng.randint(1, n - 1)
    l2 = rng.randint(1, n - l1)
    names = fast if n <= 300 else ["one_pass", "prefix_suffix", "dp", "Solution"]
    res = {name: IMPLS[name](a, l1, l2) for name in names}
    assert len(set(res.values())) == 1, (n, l1, l2, res)                                  # (1)
    ans = res["Solution"]
    if n <= 100:
        best, pos = best_positions(a, l1, l2)
        assert best == ans                                                                # (1)
        i, j = pos
        assert (i + l1 <= j or j + l2 <= i) and sum(a[i:i + l1]) + sum(a[j:j + l2]) == ans    # (7)
    assert Solution().maxSumTwoNoOverlap(a, l2, l1) == ans                                # (2)
    assert Solution().maxSumTwoNoOverlap(a[::-1], l1, l2) == ans                          # (3)
    c = rng.randint(1, 20)
    assert Solution().maxSumTwoNoOverlap([x + c for x in a], l1, l2) == ans + c * (l1 + l2)    # (4)
    assert Solution().maxSumTwoNoOverlap([x * 3 for x in a], l1, l2) == 3 * ans           # (5)
    P = [0]
    for x in a:
        P.append(P[-1] + x)
    ind = max(P[s + l1] - P[s] for s in range(n - l1 + 1)) + max(P[s + l2] - P[s] for s in range(n - l2 + 1))
    assert ans <= ind and ans >= bug_one_order(a, l1, l2)                                  # (6)
print("600 ca ngẫu nhiên (n tới 1000): 7 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

600 ca ngẫu nhiên (n tới 1000): 7 tính chất đúng; 0.5 s


In [22]:
# mảng nhỏ ngẫu nhiên: oracle định nghĩa (sum trên lát cắt) cho mọi cài đặt
rng = random.Random(103102)
t0 = time.perf_counter()
for _ in range(2500):
    n = rng.randint(2, 14)
    a = [rng.randint(0, rng.choice([2, 10, 1000])) for _ in range(n)]
    l1 = rng.randint(1, n - 1)
    l2 = rng.randint(1, n - l1)
    want = two_brute(a, l1, l2)
    for name in IMPLS:
        assert IMPLS[name](a, l1, l2) == want, (name, a, l1, l2)
print("2500 mảng nhỏ ngẫu nhiên (n ≤ 14): mọi cài đặt khớp oracle định nghĩa;", round(time.perf_counter() - t0, 1), "s")

2500 mảng nhỏ ngẫu nhiên (n ≤ 14): mọi cài đặt khớp oracle định nghĩa; 0.3 s


### F. Giới hạn

`n = 1000`, `L₁ + L₂` từ `2` tới `1000`; `L₁ = 1`, `L₂ = 999` (chỉ một cách); `L₁ = L₂ = 500`; toàn `1000`; toàn `0`; mảng tăng dần / giảm dần; ngẫu nhiên.

In [23]:
t0 = time.perf_counter()
n = 1000
rng = random.Random(103103)
arrs = {
    "ngẫu nhiên": [rng.randint(0, 1000) for _ in range(n)], "toàn 1000": [1000] * n, "toàn 0": [0] * n, "tăng dần": [i // 2 for i in range(n)], "giảm dần": [(n - i) // 2 for i in range(n)],
    "xen kẽ": [1000 if i % 2 else 0 for i in range(n)],
}
pairs = [(1, 999), (999, 1), (500, 500), (1, 1), (300, 400), (2, 998), (100, 900), (10, 20)]
for name, a in arrs.items():
    for l1, l2 in pairs:
        res = {k: IMPLS[k](a, l1, l2) for k in ("one_pass", "prefix_suffix", "dp", "Solution", "SolutionOnePass", "SolutionPrefixSuffix", "SolutionDP")}
        assert len(set(res.values())) == 1, (name, l1, l2, res)
assert Solution().maxSumTwoNoOverlap(arrs["toàn 1000"], 300, 400) == 700000 and Solution().maxSumTwoNoOverlap(arrs["toàn 0"], 1, 999) == 0
assert Solution().maxSumTwoNoOverlap(arrs["ngẫu nhiên"], 1, 999) == sum(arrs["ngẫu nhiên"])
print("n = 1000: 6 mảng × 8 cặp độ dài, các cách O(n) khớp nhau; L1 + L2 = n cho tổng cả mảng;", round(time.perf_counter() - t0, 2), "s")
sp = {k: IMPLS[k](arrs["ngẫu nhiên"], 300, 400) for k in ("sparse", "SolutionSparse")}
assert len(set(sp.values())) == 1 and sp["sparse"] == Solution().maxSumTwoNoOverlap(arrs["ngẫu nhiên"], 300, 400)
print("sparse table khớp trên n = 1000")

n = 1000: 6 mảng × 8 cặp độ dài, các cách O(n) khớp nhau; L1 + L2 = n cho tổng cả mảng; 0.1 s
sparse table khớp trên n = 1000


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ xét một thứ tự

In [24]:
a, l1, l2 = [5, 5, 5, 0, 0, 1], 1, 3
print("bug:", bug_one_order(a, l1, l2), "| đúng:", Solution().maxSumTwoNoOverlap(a, l1, l2))
assert bug_one_order(a, l1, l2) < Solution().maxSumTwoNoOverlap(a, l1, l2)

bug: 15 | đúng: 16


**Vì sao sai:** đoạn dài `firstLen` **có thể** ở sau (hoặc trước) đoạn dài `secondLen`; phải xét cả hai thứ tự (`max(solve(f, s), solve(s, f))`).

### Lỗi 2: cho phép hai cửa sổ giao nhau

In [25]:
a, l1, l2 = [0, 10, 10, 0, 0, 1], 2, 1
print("bug:", bug_overlap_allowed(a, l1, l2), "| đúng:", Solution().maxSumTwoNoOverlap(a, l1, l2))
assert bug_overlap_allowed(a, l1, l2) == 30 and Solution().maxSumTwoNoOverlap(a, l1, l2) == 21

bug: 30 | đúng: 21


**Vì sao sai:** cộng hai cửa sổ tốt nhất độc lập có thể dùng cùng một phần tử hai lần (`[10,10]` và `[10]`). **Đúng:** hai đoạn phải **rời nhau**.

### Lỗi 3: tham lam (chọn cửa sổ tốt nhất rồi đoạn còn lại)

In [26]:
bad = next((a, l1, l2) for a, l1, l2 in pool if bug_greedy_two_best(a, l1, l2) != two_brute(a, l1, l2))
print("phản ví dụ:", bad, "| bug:", bug_greedy_two_best(*bad), "| đúng:", two_brute(*bad))
assert bug_greedy_two_best(*bad) != two_brute(*bad)

phản ví dụ: ([0, 1, 0], 1, 2) | bug: 0 | đúng: 1


**Vì sao sai:** cửa sổ tốt nhất của loại thứ nhất có thể "chiếm chỗ" của cửa sổ tốt nhất loại thứ hai; tối ưu phải cân đối đồng thời.

### Lỗi 4: điều kiện không giao sai (thiếu dấu bằng)

In [27]:
bad = next((a, l1, l2) for a, l1, l2 in pool if bug_gap(a, l1, l2) != two_brute(a, l1, l2))
print("phản ví dụ:", bad, "| bug:", bug_gap(*bad), "| đúng:", two_brute(*bad))
assert bug_gap(*bad) != two_brute(*bad)

phản ví dụ: ([0, 1], 1, 1) | bug: 0 | đúng: 1


**Vì sao sai:** hai đoạn **chạm nhau** (`j == i + L₁`) vẫn không giao. Điều kiện đúng là `i + L₁ ≤ j` (không phải `<`).

### Lỗi 5: tổng cửa sổ lệch một chỉ số

In [28]:
bad = next((a, l1, l2) for a, l1, l2 in pool if bug_prefix_off_by_one(a, l1, l2) != two_brute(a, l1, l2))
print("phản ví dụ:", bad, "| bug:", bug_prefix_off_by_one(*bad), "| đúng:", two_brute(*bad))
assert bug_prefix_off_by_one(*bad) != two_brute(*bad)

phản ví dụ: ([1, 0], 1, 1) | bug: 0 | đúng: 1


**Vì sao sai:** cửa sổ độ dài `a` kết thúc tại `e` có tổng `P[e] − P[e − a]` (không phải `P[e − a + 1]`).

### Lỗi 6 (bonus): quên đổi độ dài khi đổi thứ tự

In [29]:
bad = next((a, l1, l2) for a, l1, l2 in pool if bug_not_swapped_lengths(a, l1, l2) != two_brute(a, l1, l2))
print("phản ví dụ:", bad, "| bug:", bug_not_swapped_lengths(*bad), "| đúng:", two_brute(*bad))
assert bug_not_swapped_lengths(*bad) != two_brute(*bad)

phản ví dụ: ([1, 0, 1, 1], 2, 1) | bug: 2 | đúng: 3


**Vì sao sai:** thứ tự ngược lại là **`secondLen` trước `firstLen`**: phải gọi `solve(secondLen, firstLen)`, không phải lặp lại `solve(firstLen, secondLen)`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "chọn hai (hay `k`) đoạn liên tiếp rời nhau để tối đa tổng" → tổng tiền tố + duy trì cực đại của đoạn đứng trước; xét mọi thứ tự.

**Mẹo:**

1. `P[e] − P[e − L]` là tổng cửa sổ kết thúc tại `e`.
2. Duyệt vị trí kết thúc đoạn sau; `best_a` cập nhật khi đoạn trước vừa "đóng".
3. Luôn xét **cả hai thứ tự** khi độ dài khác nhau.
4. Điều kiện rời nhau: `i + L₁ ≤ j` (chạm nhau vẫn hợp lệ).
5. Cực đại tiền tố/hậu tố hoặc sparse table cho cùng ý.
6. Oracle `O(n²)` theo định nghĩa trên mảng rất nhỏ.

**Họ bài cùng pattern:** Best Time to Buy and Sell Stock III (123), Maximum Sum of 3 Non-Overlapping Subarrays (689), Two Non-overlapping Sub-arrays Each With Target Sum (1477), Maximum Number of Non-Overlapping Subarrays With Sum Equals Target (1546), Maximum Subarray (53), Maximum Sum Circular Subarray (918).

## 15. Biến thể

1. **Trả về vị trí** của hai đoạn (đã làm ở `best_positions`).
2. **Ba đoạn** độ dài `L₁, L₂, L₃` rời nhau theo **mọi thứ tự** (`3! = 6`).
3. **Không bắt buộc đúng độ dài** (đoạn con bất kỳ): khác bài khác (Kadane hai lần).

Cài đặt ba đoạn và đối chiếu brute:

In [30]:
def three_windows(nums, lens):
    """Biến thể 2: ba cửa sổ độ dài lens[0..2] rời nhau, mọi thứ tự; DP theo vị trí kết thúc."""
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    NEG = float("-inf")
    best = NEG
    for perm in itertools.permutations(lens):
        a, b, c = perm                                    # a rồi b rồi c
        A = [NEG] * (n + 1)                               # max cửa sổ a kết thúc <= i
        AB = [NEG] * (n + 1)                              # a rồi b, b kết thúc <= i
        ABC = [NEG] * (n + 1)
        for i in range(1, n + 1):
            A[i], AB[i], ABC[i] = A[i - 1], AB[i - 1], ABC[i - 1]
            if i >= a:
                A[i] = max(A[i], P[i] - P[i - a])
            if i >= a + b and A[i - b] > NEG:
                AB[i] = max(AB[i], A[i - b] + P[i] - P[i - b])
            if i >= a + b + c and AB[i - c] > NEG:
                ABC[i] = max(ABC[i], AB[i - c] + P[i] - P[i - c])
        best = max(best, ABC[n])
    return best


def three_brute(nums, lens):
    n = len(nums)
    best = float("-inf")
    for s in itertools.product(range(n), repeat=3):
        spans = sorted(zip(s, lens))                       # mỗi cửa sổ (đầu, độ dài); thử mọi gán độ dài cho vị trí đầu
        ok = all(spans[k][0] + spans[k][1] <= spans[k + 1][0] for k in range(2)) and all(sp[0] + sp[1] <= n for sp in spans)
        if ok:
            best = max(best, sum(sum(nums[st:st + ln]) for st, ln in spans))
    return best


rng = random.Random(15)
for _ in range(300):
    n = rng.randint(3, 9)
    a = [rng.randint(0, 9) for _ in range(n)]
    lens = [rng.randint(1, 3) for _ in range(3)]
    if sum(lens) <= n:
        assert three_windows(a, lens) == three_brute(a, lens), (a, lens)
assert three_windows([1, 2, 3, 4, 5, 6], [1, 1, 1]) == 15
print("ba cửa sổ rời nhau (mọi thứ tự): khớp brute")

ba cửa sổ rời nhau (mọi thứ tự): khớp brute


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp vị trí | định nghĩa | `O(n²)` | `O(n)` | oracle |
| 2. Một lượt `best_a` | đoạn trước đã "đóng" | `O(n)` | `O(n)` | **đáp án chính** |
| 3. Tiền tố/hậu tố | ranh giới `k` | `O(n)` | `O(n)` | dạng khai báo |
| 4. Sparse table | RMQ trên cửa sổ | `O(n log n)` | `O(n log n)` | truy vấn linh hoạt |
| 5. DP trạng thái | `A, B, AB, BA` | `O(n)` | `O(n)` | mở rộng cho nhiều đoạn |

Mọi cách khớp oracle trên `8 705` mảng nhỏ × mọi cặp độ dài, và khớp nhau tới `n = 1000`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [31]:
print(Solution().maxSumTwoNoOverlap([0, 6, 5, 2, 2, 5, 1, 9, 4], 1, 2), Solution().maxSumTwoNoOverlap([3, 8, 1, 3, 2, 1, 8, 9, 0], 3, 2),
      Solution().maxSumTwoNoOverlap([2, 1, 5, 6, 0, 9, 5, 0, 3, 8], 4, 3), Solution().maxSumTwoNoOverlap([1, 1], 1, 1), Solution().maxSumTwoNoOverlap([5, 0, 0, 5], 2, 2))

20 29 31 2 10


### Bài 2 (tính tay)

`nums = [1, 4, 2, 9, 3, 7, 5]`, `firstLen = 2`, `secondLen = 3`. (a) Các cửa sổ độ dài 2 và 3 với tổng. (b) Cặp không giao tốt nhất. (c) Đáp án. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hàm `solve(a, b)` (đoạn dài `a` trước đoạn dài `b`):

In [32]:
def solve_todo(nums, a, b):
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    best_a = ans = 0
    for j in range(a + b, len(nums) + 1):
        pass            # TODO 1: best_a = max(best_a, P[j - b] - P[j - b - a])
        pass            # TODO 2: ans = max(ans, best_a + P[j] - P[j - b])
    return ans


# Khi làm xong, bỏ comment để kiểm tra:
# assert max(solve_todo(examples[0][0][0], 1, 2), solve_todo(examples[0][0][0], 2, 1)) == 20

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **mảng cực đại tiền tố / hậu tố** của tổng cửa sổ (xét cả hai thứ tự):

In [33]:
class MySolution:
    def maxSumTwoNoOverlap(self, nums, firstLen, secondLen):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().maxSumTwoNoOverlap(list(a), l1, l2) == w for (a, l1, l2), w in examples)

### Bài 5 (mini challenge): ba đoạn có độ dài `L`

Cho `nums` và `L`, trả về tổng lớn nhất của **ba** đoạn con **rời nhau**, mỗi đoạn dài đúng `L` (LeetCode 689 chỉ trả tổng tối đa; ở đây chỉ cần giá trị). Gợi ý: DP `A`, `AB`, `ABC` theo vị trí kết thúc. Kiểm bằng `three_brute`.

In [34]:
def three_equal(nums, L):
    pass            # TODO


# Khi làm xong:
# assert three_equal([1, 2, 1, 2, 6, 7, 5, 1], 2) == 23

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: ví dụ 1–3 cho `20`, `29`, `31`.
- Mức 2: `[1,1]` với `(1,1)` ⇒ `2`; `[5,0,0,5]` với `(2,2)`: `[5,0]` và `[0,5]` ⇒ `10`.
- Mức 3: `20 29 31 2 10`.

**Bài 2**

- Mức 1: cửa sổ độ dài 2: `5, 6, 11, 12, 10, 12`; độ dài 3: `7, 15, 14, 19, 15`.
- Mức 2: ví dụ `[9,3,7]` (độ dài 3, tổng 19) và `[1,4]` (độ dài 2, tổng 5) ⇒ 24; `[4,2,9]` (15) với `[7,5]` (12) ⇒ 27.
- Mức 3: đáp án `27` (đoạn dài 3 ở vị trí `1`, đoạn dài 2 ở vị trí `5`).

**Bài 3**

- Mức 1: `best_a = max(best_a, P[j - b] - P[j - b - a])`.
- Mức 2: `ans = max(ans, best_a + P[j] - P[j - b])`.
- Mức 3: kết hợp hai thứ tự bằng `max`.

**Bài 4**

- Mức 1: `before[k]`: max cửa sổ `a` nằm trong `[0, k)`.
- Mức 2: `after[k]`: max cửa sổ `b` nằm trong `[k, n)`.
- Mức 3: `max(before[k] + after[k])` cho `k` từ `a` đến `n - b`; làm cả `(a, b)` và `(b, a)`.

**Bài 5**

- Mức 1: `A[i]`, `AB[i]`, `ABC[i]` theo độ dài tiền tố.
- Mức 2: `AB[i] = max(AB[i-1], A[i-L] + window(i))`, tương tự `ABC`.
- Mức 3: trả `ABC[n]`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [35]:
vals = (Solution().maxSumTwoNoOverlap([0, 6, 5, 2, 2, 5, 1, 9, 4], 1, 2), Solution().maxSumTwoNoOverlap([3, 8, 1, 3, 2, 1, 8, 9, 0], 3, 2),
        Solution().maxSumTwoNoOverlap([2, 1, 5, 6, 0, 9, 5, 0, 3, 8], 4, 3), Solution().maxSumTwoNoOverlap([1, 1], 1, 1), Solution().maxSumTwoNoOverlap([5, 0, 0, 5], 2, 2))
print(vals)
assert vals == (20, 29, 31, 2, 10)
print("Bài 1 OK")

(20, 29, 31, 2, 10)
Bài 1 OK


### Bài 2

In [36]:
a = [1, 4, 2, 9, 3, 7, 5]
P = [0]
for x in a:
    P.append(P[-1] + x)
w2 = [P[s + 2] - P[s] for s in range(len(a) - 1)]
w3 = [P[s + 3] - P[s] for s in range(len(a) - 2)]
print(w2, w3)
assert w2 == [5, 6, 11, 12, 10, 12] and w3 == [7, 15, 14, 19, 15]
ans = Solution().maxSumTwoNoOverlap(a, 2, 3)
best, pos = best_positions(a, 2, 3)
print(ans, pos)
assert ans == 27 == two_brute(a, 2, 3) == best and pos == (5, 1)
print("Bài 2 OK: 27")

[5, 6, 11, 12, 10, 12] [7, 15, 14, 19, 15]
27 (5, 1)
Bài 2 OK: 27


### Bài 3

In [37]:
def solve_answer(nums, a, b):
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    best_a = ans = 0
    for j in range(a + b, len(nums) + 1):
        best_a = max(best_a, P[j - b] - P[j - b - a])
        ans = max(ans, best_a + P[j] - P[j - b])
    return ans


assert max(solve_answer(examples[0][0][0], 1, 2), solve_answer(examples[0][0][0], 2, 1)) == 20
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(2, 12)
    arr = [rng.randint(0, 9) for _ in range(n)]
    l1 = rng.randint(1, n - 1)
    l2 = rng.randint(1, n - l1)
    assert max(solve_answer(arr, l1, l2), solve_answer(arr, l2, l1)) == two_brute(arr, l1, l2)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def maxSumTwoNoOverlap(self, nums, firstLen, secondLen):
        n = len(nums)
        P = [0]
        for x in nums:
            P.append(P[-1] + x)
        NEG = float("-inf")

        def order(a, b):
            before = [NEG] * (n + 1)
            for k in range(a, n + 1):
                before[k] = max(before[k - 1], P[k] - P[k - a])
            after = [NEG] * (n + 2)
            for k in range(n - b, -1, -1):
                after[k] = max(after[k + 1], P[k + b] - P[k])
            return max(before[k] + after[k] for k in range(a, n - b + 1))

        return max(order(firstLen, secondLen), order(secondLen, firstLen))


assert all(MySolutionAnswer().maxSumTwoNoOverlap(list(a), l1, l2) == w for (a, l1, l2), w in examples)
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(2, 12)
    arr = [rng.randint(0, 9) for _ in range(n)]
    l1 = rng.randint(1, n - 1)
    l2 = rng.randint(1, n - l1)
    assert MySolutionAnswer().maxSumTwoNoOverlap(arr, l1, l2) == two_brute(arr, l1, l2)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def three_equal(nums, L):
    return three_windows(nums, [L, L, L])


assert three_equal([1, 2, 1, 2, 6, 7, 5, 1], 2) == 23
rng = random.Random(5)
for _ in range(200):
    n = rng.randint(3, 9)
    L = rng.randint(1, max(1, n // 3))
    arr = [rng.randint(0, 9) for _ in range(n)]
    assert three_equal(arr, L) == three_brute(arr, [L, L, L])
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (hai đoạn liên tiếp rời nhau tối đa tổng → tổng tiền tố + cực đại của đoạn đứng trước, cả hai thứ tự):**

```python
class Solution:
    def maxSumTwoNoOverlap(self, nums: List[int], firstLen: int, secondLen: int) -> int:
        n = len(nums)
        P = [0]
        for x in nums:
            P.append(P[-1] + x)

        def solve(a, b):                       # đoạn dài a đứng TRƯỚC đoạn dài b
            best_a = ans = 0
            for j in range(a + b, n + 1):
                best_a = max(best_a, P[j - b] - P[j - b - a])
                ans = max(ans, best_a + P[j] - P[j - b])
            return ans

        return max(solve(firstLen, secondLen), solve(secondLen, firstLen))
```

**Checklist trước khi submit:**

- [ ] Xét **cả hai thứ tự** (`firstLen` trước và `secondLen` trước)?
- [ ] Điều kiện rời nhau: đoạn trước kết thúc tại hoặc trước chỗ đoạn sau bắt đầu (chạm nhau vẫn hợp lệ)?
- [ ] Tổng cửa sổ `P[e] − P[e − L]` (không lệch chỉ số)?
- [ ] Không chọn tham lam hai cửa sổ tốt nhất độc lập (có thể giao nhau)?
- [ ] Vòng `j` bắt đầu từ `a + b`?
- [ ] Đã thử ví dụ 1–3, `[5,0,0,5]` với `(2,2)`, `n = firstLen + secondLen`?
- [ ] Đúng tên `class Solution`, `maxSumTwoNoOverlap(self, nums, firstLen, secondLen)`, không in debug.